# 🕵️ Diagnostic Avancé de Dérive (Drift)

Ce notebook analyse si les données de Test (`conversion_data_test.csv`) sont fondamentalement différentes des données d'Entraînement (`conversion_data_train.csv`).

**Méthodologie :**
1.  **Comparaison Visuelle** : Distributions des features clés.
2.  **Métriques Statistiques** : Distance de Wasserstein (Numérique) & PSI (Catégoriel).
3.  **Adversarial Validation** : On entraîne un modèle pour essayer de distinguer Train vs Test. Si le modèle échoue (AUC ~ 0.50), c'est qu'il n'y a pas de dérive. S'il réussit (AUC > 0.70), il y a un problème.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wasserstein_distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement des Données

In [ ]:
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

# Ajout d'une colonne 'is_test' pour l'Adversarial Validation
train_df['is_test'] = 0
test_df['is_test'] = 1

# Feature Engineering minimaliste (pour voir si le drift vient des ratios)
for df in [train_df, test_df]:
    df['pages_per_age'] = df['total_pages_visited'] / (df['age'] + 0.1)

## 2. Comparaison Visuelle & Statistique
On regarde les distributions et on calcule la **Distance de Wasserstein** (plus elle est petite, mieux c'est).

In [ ]:
features_num = ['age', 'total_pages_visited', 'pages_per_age']
features_cat = ['country', 'source', 'new_user']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# --- Numériques ---
for i, col in enumerate(features_num):
    ax = axes[i]
    sns.kdeplot(train_df[col], label='Train', fill=True, ax=ax, color='blue', alpha=0.3)
    sns.kdeplot(test_df[col], label='Test', fill=True, ax=ax, color='orange', alpha=0.3)
    wd = wasserstein_distance(train_df[col], test_df[col])
    ax.set_title(f"{col} (W-dist: {wd:.4f})")
    ax.legend()

# --- Catégorielles ---
for j, col in enumerate(features_cat):
    idx = len(features_num) + j
    ax = axes[idx]
    
    # Calcul des proportions
    train_props = train_df[col].value_counts(normalize=True).sort_index()
    test_props = test_df[col].value_counts(normalize=True).sort_index()
    
    df_plot = pd.DataFrame({'Train': train_props, 'Test': test_props})
    df_plot.plot(kind='bar', ax=ax, alpha=0.7, color=['blue', 'orange'])
    ax.set_title(f"{col} Distribution")

plt.tight_layout()
plt.show()

## 3. Adversarial Validation (Le Juge de Paix)
On entraîne un classifieur pour distinguer Train vs Test.
*   **AUC ~ 0.50** : Indiscernables = **Pas de Drift** (Parfait).
*   **AUC > 0.70** : Discernables = **Drift Important** (Danger).

In [ ]:
# Préparation
full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
target_adv = full_df['is_test']
X_adv = full_df.drop(['converted', 'is_test'], axis=1)

# Encoding
for col in ['country', 'source']:
    le = LabelEncoder()
    X_adv[col] = le.fit_transform(X_adv[col])

# Modèle Détecteur
clf_adv = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

# Cross-Validation ROC AUC
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf_adv, X_adv, target_adv, cv=cv, scoring='roc_auc')

print(f"🛡️ Adversarial AUC Score : {scores.mean():.4f} (+/- {scores.std():.4f})")

if scores.mean() < 0.60:
    print("✅ CONCLUSION : Peu ou pas de dérive. Le modèle fonctionne en terrain connu.")
else:
    print("⚠️ CONCLUSION : Dérive détectée ! Le Test est différent du Train.")
    
# Feature Importance (si drift, d'où vient-il ?)
clf_adv.fit(X_adv, target_adv)
imp = pd.Series(clf_adv.feature_importances_, index=X_adv.columns).sort_values(ascending=False)
print("\nTop Features responsables du Drift :")
print(imp.head(3))